<a href="https://colab.research.google.com/github/rachelwhite14/Game-Investment-Analyzer/blob/main/game_investment_analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Game Investment analyzer
My app helps smaller game developers decide what game genre to invest in by analyzing genre popularity, player satisfaction and market pricing found in the Steam data.

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/MIST 4600/


In [3]:
games_by_genre = games.copy()
games_by_genre["genres"] = games_by_genre["genres"].str.split(", ")
games_by_genre = games_by_genre.explode("genres")

In [4]:
def best_genres(games, n):
    avg_owners = games.groupby("genres")["owners_low"].mean()
    num_games = games["genres"].value_counts()

    genres = pd.DataFrame({
        "avg_owners": avg_owners,
        "num_games": num_games
    })

    genres["score"] = genres["avg_owners"] / genres["num_games"]

    best = genres.sort_values("score", ascending=False)

    return best.head(n)

print(best_genres(games_by_genre, 5))

                         avg_owners  num_games         score
genres                                                      
Sports                 2.723077e+06         39  69822.485207
Racing                 1.925806e+06         31  62122.788762
Massively Multiplayer  4.579630e+06        108  42403.978052
Early Access           2.830435e+06         69  41020.793951
Free To Play           5.067725e+06        189  26813.359088


In [5]:
print(games.columns)

Index(['appid', 'name', 'release_date', 'release_year', 'price', 'discount',
       'required_age', 'windows', 'mac', 'linux', 'developers', 'publishers',
       'genres', 'n_languages', 'achievements', 'metacritic_score', 'positive',
       'negative', 'pct_pos_total', 'num_reviews_total', 'estimated_owners',
       'owners_low', 'avg_playtime_hours', 'peak_ccu', 'short_description',
       'critic_reviews'],
      dtype='object')


In [6]:
def best_reviewed_genres(games, n):
    # Find the average positive review percentage for each genre
    avg_reviews = games.groupby("genres")["pct_pos_total"].mean()

    # Sort from highest percentage to lowest
    best = avg_reviews.sort_values(ascending=False)

    # Return the top n genres
    return best.head(n)

print(best_reviewed_genres(games_by_genre, 5))

genres
Indie         89.648019
Casual        88.630137
Simulation    86.734375
Adventure     86.171765
Strategy      85.068293
Name: pct_pos_total, dtype: float64


In [7]:
def highest_priced_genres(games, n):
    # Find the median selling price for each genre
    typical_price = games.groupby("genres")["price"].median()

    # Sort from highest price to lowest
    best = typical_price.sort_values(ascending=False)

    # Return the top n genres
    return best.head(n)

print(highest_priced_genres(games_by_genre, 5))

genres
Action       19.99
Adventure    19.99
Sports       19.99
RPG          19.99
Strategy     19.99
Name: price, dtype: float64


In [8]:
class GameInvestmentAnalyzer:
    def __init__(self, games):
        self.games = games

    def show_games(self):
        print("Total games analyzed:", len(self.games))


# Create an object using our class
analyzer = GameInvestmentAnalyzer(games)

# Call the method
analyzer.show_games()

Total games analyzed: 1000


In [9]:
!pip install -q google-genai

In [16]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say hello!"
)

print(response.text)

Hello! How can I help you today?


In [17]:
# Get our results from sub-question 1
top_genres = best_genres(games_by_genre, 5)

# Ask Gemini to analyze the results
prompt = f"""
I am a game developer deciding which Steam game genre
to invest in.

Here are my top five genres ranked by demand
relative to competition:

{top_genres.to_string()}

Which genre looks most promising based on these numbers?
Explain your answer in 3 simple sentences.
Mention one limitation of the data.
"""

# Send the question to Gemini
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

# Print Gemini's recommendation
print(response.text)


**Sports** looks like the most promising genre because it has the highest score of 69,822, indicating the best balance of high player demand relative to low competition. It achieves an impressive average of over 2.7 million owners while maintaining the smallest number of competing games (39) on the list. This combination suggests an underserved market with high potential for a new game to capture an eager audience. 

**Limitation of the data:** The dataset lacks revenue or monetization data, meaning high owner counts might just reflect the popularity of free games rather than actual profit potential.


In [18]:
# Get the results from sub-question 2
top_reviews = best_reviewed_genres(games_by_genre, 5)

# Ask Gemini to interpret the results
prompt = f"""
I am a game developer deciding which Steam game genre
to invest in.

These are the genres with the highest average
positive review percentages:

{top_reviews.to_string()}

Which genres have the highest customer satisfaction?
Explain what these results mean for a game developer
in 3 simple sentences.
Only use the data provided.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)


Based on the provided data, Indie and Casual games have the highest customer satisfaction, with positive review percentages of 89.65% and 88.63% respectively. For a game developer, these results mean that investing in Indie or Casual genres yields the highest average positive review percentages among the listed options. Therefore, prioritizing these specific genres offers the greatest likelihood of achieving high customer satisfaction based on the provided metrics.


In [19]:
# Get the results from sub-question 3
top_prices = highest_priced_genres(games_by_genre, 5)

# Ask Gemini to interpret the results
prompt = f"""
I am a game developer deciding which Steam game genre
to invest in.

These are the genres with the highest median
selling prices:

{top_prices.to_string()}

What do these prices suggest about potential
revenue opportunities for a game developer?
Explain in 3 simple sentences.
Only use the data provided.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)


The data shows that Action, Adventure, Sports, RPG, and Strategy all share the exact same highest median selling price of 19.99. This suggests that a game developer can potentially charge the same price point regardless of which of these five specific genres they choose to invest in. However, because the data only provides median selling prices and not sales volume, actual potential revenue opportunities cannot be fully determined from price alone.


In [20]:
sports_reviews = games_by_genre[games_by_genre["genres"] == "Sports"]["pct_pos_total"].mean()

print("Sports average positive review percentage:", round(sports_reviews, 2))


Sports average positive review percentage: 76.15
